In [3]:
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

In [5]:
def process_all_pdfs(pdf_directory):
    """Process all PDF files in a directory"""
    all_documents = []
    pdf_dir = Path(pdf_directory)
    
    # Find all PDF files recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    
    print(f"Found {len(pdf_files)} PDF files to process")
    
    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()
            
            # Add source information to metadata
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'
            
            all_documents.extend(documents)
            print(f"  ✓ Loaded {len(documents)} pages")
            
        except Exception as e:
            print(f"  ✗ Error: {e}")
    
    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents

# Process all PDFs in the data directory
all_pdf_documents = process_all_pdfs("../data")


Found 3 PDF files to process

Processing: U2 and U4.pdf
  ✓ Loaded 174 pages

Processing: UNIT 2.pdf
  ✓ Loaded 4 pages

Processing: attension.pdf
  ✓ Loaded 11 pages

Total documents loaded: 189


In [6]:
all_pdf_documents

[Document(metadata={'producer': 'macOS Version 26.3.1 (a) (Build 25D771280a) Quartz PDFContext, AppendMode 1.1', 'creator': 'Writer', 'creationdate': '2026-06-11T21:06:15+00:00', 'moddate': "D:20260728093339Z00'00'", 'source': '../data/pdf/U2 and U4.pdf', 'total_pages': 174, 'page': 0, 'page_label': '1', 'source_file': 'U2 and U4.pdf', 'file_type': 'pdf'}, page_content="Analyze these papers and say 'yes'   when ready\nYes. ✅\nI've analyzed both papers:\n\uf0b7 AI End Sem 2025 \n AI End Sem 2025\n\uf0b7 AI End Sem 2024 \n AI endsem 2024\nI have a good understanding of:\n\uf0b7 Unit-wise question distribution\n\uf0b7 Repeated topics across years\n\uf0b7 Important long-answer vs short-answer trends\n\uf0b7 Frequently asked algorithms (A*, Alpha-Beta, CSP, Resolution, Bayesian \nNetworks, Expert Systems, K-Means, etc.)\n\uf0b7 High-probability questions for the next exam based on recurrence patterns\nReady for the next step. 🚀\nYou can now ask me to:\n1. Create a 70-80% accurate guess pape

In [7]:
def split_documents(documents,chunk_size=1000,chunk_overlap=200):
    """Split documents into smaller chunks for better RAG performance"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")
    
    # Show example of a chunk
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")
    
    return split_docs

In [8]:
chunks=split_documents(all_pdf_documents)
chunks

Split 189 documents into 218 chunks

Example chunk:
Content: Analyze these papers and say 'yes'   when ready
Yes. ✅
I've analyzed both papers:
 AI End Sem 2025 
 AI End Sem 2025
 AI End Sem 2024 
 AI endsem 2024
I have a good understanding of:
 Unit-wise que...
Metadata: {'producer': 'macOS Version 26.3.1 (a) (Build 25D771280a) Quartz PDFContext, AppendMode 1.1', 'creator': 'Writer', 'creationdate': '2026-06-11T21:06:15+00:00', 'moddate': "D:20260728093339Z00'00'", 'source': '../data/pdf/U2 and U4.pdf', 'total_pages': 174, 'page': 0, 'page_label': '1', 'source_file': 'U2 and U4.pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': 'macOS Version 26.3.1 (a) (Build 25D771280a) Quartz PDFContext, AppendMode 1.1', 'creator': 'Writer', 'creationdate': '2026-06-11T21:06:15+00:00', 'moddate': "D:20260728093339Z00'00'", 'source': '../data/pdf/U2 and U4.pdf', 'total_pages': 174, 'page': 0, 'page_label': '1', 'source_file': 'U2 and U4.pdf', 'file_type': 'pdf'}, page_content="Analyze these papers and say 'yes'   when ready\nYes. ✅\nI've analyzed both papers:\n\uf0b7 AI End Sem 2025 \n AI End Sem 2025\n\uf0b7 AI End Sem 2024 \n AI endsem 2024\nI have a good understanding of:\n\uf0b7 Unit-wise question distribution\n\uf0b7 Repeated topics across years\n\uf0b7 Important long-answer vs short-answer trends\n\uf0b7 Frequently asked algorithms (A*, Alpha-Beta, CSP, Resolution, Bayesian \nNetworks, Expert Systems, K-Means, etc.)\n\uf0b7 High-probability questions for the next exam based on recurrence patterns\nReady for the next step. 🚀\nYou can now ask me to:\n1. Create a 70-80% accurate guess pape

## Embeddings and VectorDB

In [19]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [20]:
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""
    
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        Initialize the embedding manager
        
        Args:
            model_name: HuggingFace model name for sentence embeddings
        """
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the SentenceTransformer model"""
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of texts
        
        Args:
            texts: List of text strings to embed
            
        Returns:
            numpy array of embeddings with shape (len(texts), embedding_dim)
        """
        if not self.model:
            raise ValueError("Model not loaded")
        
        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings

## initialize the embedding manager

embedding_manager=EmbeddingManager()
embedding_manager

Loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6819.79it/s]


Model loaded successfully. Embedding dimension: 384


/var/folders/98/1c3r39wd7jb3mkdhxs1_9b1c0000gn/T/ipykernel_41338/3630282658.py:20: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")


### Vector Store

In [21]:
class VectorStore:
    """Manages document embeddings in a ChromaDB vector store"""
    
    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        """
        Initialize the vector store
        
        Args:
            collection_name: Name of the ChromaDB collection
            persist_directory: Directory to persist the vector store
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""
        try:
            # Create persistent ChromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)
            
            # Get or create collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF document embeddings for RAG"}
            )
            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store
        
        Args:
            documents: List of LangChain documents
            embeddings: Corresponding embeddings for the documents
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")
        
        print(f"Adding {len(documents)} documents to vector store...")
        
        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []
        
        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)
            
            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)
            
            # Document content
            documents_text.append(doc.page_content)
            
            # Embedding
            embeddings_list.append(embedding.tolist())
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise

vectorstore=VectorStore()
vectorstore
    

Vector store initialized. Collection: pdf_documents
Existing documents in collection: 218


In [22]:
chunks

[Document(metadata={'producer': 'macOS Version 26.3.1 (a) (Build 25D771280a) Quartz PDFContext, AppendMode 1.1', 'creator': 'Writer', 'creationdate': '2026-06-11T21:06:15+00:00', 'moddate': "D:20260728093339Z00'00'", 'source': '../data/pdf/U2 and U4.pdf', 'total_pages': 174, 'page': 0, 'page_label': '1', 'source_file': 'U2 and U4.pdf', 'file_type': 'pdf'}, page_content="Analyze these papers and say 'yes'   when ready\nYes. ✅\nI've analyzed both papers:\n\uf0b7 AI End Sem 2025 \n AI End Sem 2025\n\uf0b7 AI End Sem 2024 \n AI endsem 2024\nI have a good understanding of:\n\uf0b7 Unit-wise question distribution\n\uf0b7 Repeated topics across years\n\uf0b7 Important long-answer vs short-answer trends\n\uf0b7 Frequently asked algorithms (A*, Alpha-Beta, CSP, Resolution, Bayesian \nNetworks, Expert Systems, K-Means, etc.)\n\uf0b7 High-probability questions for the next exam based on recurrence patterns\nReady for the next step. 🚀\nYou can now ask me to:\n1. Create a 70-80% accurate guess pape

In [23]:
### Convert the text to embeddings
texts=[doc.page_content for doc in chunks]

## Generate the Embeddings

embeddings=embedding_manager.generate_embeddings(texts)

##store int he vector dtaabase
vectorstore.add_documents(chunks,embeddings)

Generating embeddings for 218 texts...


Batches: 100%|██████████| 7/7 [00:00<00:00, 10.75it/s]


Generated embeddings with shape: (218, 384)
Adding 218 documents to vector store...
Successfully added 218 documents to vector store
Total documents in collection: 436


### Retriever Pipeline From VectorStore

In [43]:

class RAGRetriever:
    """Handles query-based retrieval from the vector store"""
    
    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """
        Initialize the retriever
        
        Args:
            vector_store: Vector store containing document embeddings
            embedding_manager: Manager for generating query embeddings
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents for a query
        
        Args:
            query: The search query
            top_k: Number of top results to return
            score_threshold: Minimum similarity score threshold
            
        Returns:
            List of dictionaries containing retrieved documents and metadata
        """
        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")
        
        # Generate query embedding
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]
        
        # Search in vector store
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )
            print(results)
            
            # Process results
            retrieved_docs = []
            
            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]
                
                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    retrieved_docs.append({
                        'id': doc_id,
                        'content': document,
                        'metadata': metadata,
                        'distance': distance,
                        'rank': i + 1
                    })
                
                print(f"Retrieved {len(retrieved_docs)} documents")
            else:
                print("No documents found")
            
            return retrieved_docs
            
        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []

rag_retriever=RAGRetriever(vectorstore,embedding_manager)

In [36]:
rag_retriever

In [38]:
rag_retriever.retrieve("What is attention is all you need")

Retrieving documents for query: 'What is attention is all you need'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 23.02it/s]

Generated embeddings with shape: (1, 384)
{'ids': [['doc_933d5e36_200', 'doc_92342847_200', 'doc_6dca4359_196', 'doc_036d94ad_196', 'doc_d3b928b2_187']], 'embeddings': None, 'documents': [['convolution is equal to the combination of a self-attention layer and a point-wise feed-forward layer,\nthe approach we take in our model.\nAs side beneﬁt, self-attention could yield more interpretable models. We inspect attention distributions\nfrom our models and present and discuss examples in the appendix. Not only do individual attention\nheads clearly learn to perform different tasks, many appear to exhibit behavior related to the syntactic\nand semantic structure of the sentences.\n5 Training\nThis section describes the training regime for our models.\n5.1 Training Data and Batching\nWe trained on the standard WMT 2014 English-German dataset consisting of about 4.5 million\nsentence pairs. Sentences were encoded using byte-pair encoding [ 3], which has a shared source-\ntarget vocabulary of a

[]

In [26]:
rag_retriever.retrieve("What paper is it")

Retrieving documents for query: 'What paper is it'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:01<00:00,  1.09s/it]

Generated embeddings with shape: (1, 384)
Retrieved 0 documents (after filtering)


[]

In [30]:
rag_retriever.retrieve("what will come in ai endsem")

Retrieving documents for query: 'what will come in ai endsem'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 27.30it/s]

Generated embeddings with shape: (1, 384)
Retrieved 0 documents (after filtering)


[]

In [31]:
print(vectorstore.collection.count())

436


In [37]:
print("Before")

results = rag_retriever.retrieve("machine learning")

print("After")
print(results)

Before
Retrieving documents for query: 'machine learning'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00,  2.26it/s]

Generated embeddings with shape: (1, 384)
{'ids': [['doc_5616eef6_41', 'doc_f29d7c0c_41', 'doc_56e0f2da_59', 'doc_e5f34880_59', 'doc_a4d8e67e_61']], 'embeddings': None, 'documents': [["all use Machine Learning.\nFirst Understand the Problem\nTraditional Programming:\nInput + Rules\n       ↓\n    Output\nExample:\nMarks > 40\nPass\nMarks < 40\nFail\nYou manually write rules.\nBut what if there are millions of rules?\nExample:\nIdentify Dog\nIdentify Cat\nIdentify Human Face\nIdentify Cancer\nImpossible to manually write every rule.\nSo instead:\n We let the computer learn from data.👉\nThat's Machine Learning.", "all use Machine Learning.\nFirst Understand the Problem\nTraditional Programming:\nInput + Rules\n       ↓\n    Output\nExample:\nMarks > 40\nPass\nMarks < 40\nFail\nYou manually write rules.\nBut what if there are millions of rules?\nExample:\nIdentify Dog\nIdentify Cat\nIdentify Human Face\nIdentify Cancer\nImpossible to manually write every rule.\nSo instead:\n We let the com

In [45]:
rag_retriever.retrieve("what is ml")

Retrieving documents for query: 'what is ml'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00,  3.25it/s]

Generated embeddings with shape: (1, 384)
{'ids': [['doc_841e7e05_49', 'doc_16e092b8_49', 'doc_de835eb0_48', 'doc_e256799b_48', 'doc_5fc303ab_43']], 'embeddings': None, 'documents': [['Exam Hack 🎯\nIf they ask:\n"What is Machine Learning?"\nAlways write:\nDefinition\nTypes of ML\nApplications\nAdvantages\nEven if not explicitly asked.\nProfs love it.\nUltra-Short Revision ⚡\nMachine Learning\n=\nLearning from Data\nTypes:\n1. Supervised\n2. Unsupervised\n3. Reinforcement\nNow only Perception and Action is left in Unit 4.\nGood news: it\'s one of the shortest topics in the entire syllabus and can be \nfinished in 10 minutes. 🚀\nlessgo', 'Exam Hack 🎯\nIf they ask:\n"What is Machine Learning?"\nAlways write:\nDefinition\nTypes of ML\nApplications\nAdvantages\nEven if not explicitly asked.\nProfs love it.\nUltra-Short Revision ⚡\nMachine Learning\n=\nLearning from Data\nTypes:\n1. Supervised\n2. Unsupervised\n3. Reinforcement\nNow only Perception and Action is left in Unit 4.\nGood news: i

[{'id': 'doc_841e7e05_49',
  'content': 'Exam Hack 🎯\nIf they ask:\n"What is Machine Learning?"\nAlways write:\nDefinition\nTypes of ML\nApplications\nAdvantages\nEven if not explicitly asked.\nProfs love it.\nUltra-Short Revision ⚡\nMachine Learning\n=\nLearning from Data\nTypes:\n1. Supervised\n2. Unsupervised\n3. Reinforcement\nNow only Perception and Action is left in Unit 4.\nGood news: it\'s one of the shortest topics in the entire syllabus and can be \nfinished in 10 minutes. 🚀\nlessgo',
  'metadata': {'producer': 'macOS Version 26.3.1 (a) (Build 25D771280a) Quartz PDFContext, AppendMode 1.1',
   'page': 48,
   'creationdate': '2026-06-11T21:06:15+00:00',
   'moddate': "D:20260728093339Z00'00'",
   'creator': 'Writer',
   'total_pages': 174,
   'source': '../data/pdf/U2 and U4.pdf',
   'file_type': 'pdf',
   'page_label': '49',
   'source_file': 'U2 and U4.pdf',
   'content_length': 434,
   'doc_index': 49},
  'distance': 1.1696357727050781,
  'rank': 1},
 {'id': 'doc_16e092b8_4

### RAG Pipeline- VectorDB To LLM Output Generation


In [50]:
import os
from dotenv import load_dotenv
load_dotenv()


from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate
from langchain_core.messages import HumanMessage, SystemMessage

In [51]:
class GroqLLM:
    def __init__(self, model_name: str = "gemma2-9b-it", api_key: str =None):
        """
        Initialize Groq LLM
        
        Args:
            model_name: Groq model name (qwen2-72b-instruct, llama3-70b-8192, etc.)
            api_key: Groq API key (or set GROQ_API_KEY environment variable)
        """
        self.model_name = model_name
        self.api_key = api_key or os.environ.get("GROQ_API_KEY")
        
        if not self.api_key:
            raise ValueError("Groq API key is required. Set GROQ_API_KEY environment variable or pass api_key parameter.")
        
        self.llm = ChatGroq(
            groq_api_key=self.api_key,
            model_name=self.model_name,
            temperature=0.1,
            max_tokens=1024
        )
        
        print(f"Initialized Groq LLM with model: {self.model_name}")

    def generate_response(self, query: str, context: str, max_length: int = 500) -> str:
        """
        Generate response using retrieved context
        
        Args:
            query: User question
            context: Retrieved document context
            max_length: Maximum response length
            
        Returns:
            Generated response string
        """
        
        # Create prompt template
        prompt_template = PromptTemplate(
            input_variables=["context", "question"],
            template="""You are a helpful AI assistant. Use the following context to answer the question accurately and concisely.

Context:
{context}

Question: {question}

Answer: Provide a clear and informative answer based on the context above. If the context doesn't contain enough information to answer the question, say so."""
        )
        
        # Format the prompt
        formatted_prompt = prompt_template.format(context=context, question=query)
        
        try:
            # Generate response
            messages = [HumanMessage(content=formatted_prompt)]
            response = self.llm.invoke(messages)
            return response.content
            
        except Exception as e:
            return f"Error generating response: {str(e)}"
        
    def generate_response_simple(self, query: str, context: str) -> str:
        """
        Simple response generation without complex prompting
        
        Args:
            query: User question
            context: Retrieved context
            
        Returns:
            Generated response
        """
        simple_prompt = f"""Based on this context: {context}

Question: {query}

Answer:"""
        
        try:
            messages = [HumanMessage(content=simple_prompt)]
            response = self.llm.invoke(messages)
            return response.content
        except Exception as e:
            return f"Error: {str(e)}"
    
        
    

In [52]:
# Initialize Groq LLM (you'll need to set GROQ_API_KEY environment variable)
try:
    groq_llm = GroqLLM(api_key=os.getenv("GROQ_API_KEY"))
    print("Groq LLM initialized successfully!")
except ValueError as e:
    print(f"Warning: {e}")
    print("Please set your GROQ_API_KEY environment variable to use the LLM.")
    groq_llm = None

Initialized Groq LLM with model: gemma2-9b-it
Groq LLM initialized successfully!


In [53]:
rag_retriever.retrieve("Unified Multi-task Learning Framework")

Retrieving documents for query: 'Unified Multi-task Learning Framework'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00,  2.18it/s]

Generated embeddings with shape: (1, 384)
{'ids': [['doc_48915df1_214', 'doc_08a9a1e8_214', 'doc_d3b928b2_187', 'doc_f6d26311_187', 'doc_ec82798e_212']], 'embeddings': None, 'documents': [['2017.\n[16] Yoon Kim, Carl Denton, Luong Hoang, and Alexander M. Rush. Structured attention networks.\nIn International Conference on Learning Representations , 2017.\n[17] Diederik Kingma and Jimmy Ba. Adam: A method for stochastic optimization. In ICLR, 2015.\n[18] Oleksii Kuchaiev and Boris Ginsburg. Factorization tricks for LSTM networks. arXiv preprint\narXiv:1703.10722, 2017.\n[19] Zhouhan Lin, Minwei Feng, Cicero Nogueira dos Santos, Mo Yu, Bing Xiang, Bowen\nZhou, and Yoshua Bengio. A structured self-attentive sentence embedding. arXiv preprint\narXiv:1703.03130, 2017.\n[20] Samy Bengio Łukasz Kaiser. Can active memory replace attention? In Advances in Neural\nInformation Processing Systems, (NIPS), 2016.\n10', '2017.\n[16] Yoon Kim, Carl Denton, Luong Hoang, and Alexander M. Rush. Structure

[{'id': 'doc_48915df1_214',
  'content': '2017.\n[16] Yoon Kim, Carl Denton, Luong Hoang, and Alexander M. Rush. Structured attention networks.\nIn International Conference on Learning Representations , 2017.\n[17] Diederik Kingma and Jimmy Ba. Adam: A method for stochastic optimization. In ICLR, 2015.\n[18] Oleksii Kuchaiev and Boris Ginsburg. Factorization tricks for LSTM networks. arXiv preprint\narXiv:1703.10722, 2017.\n[19] Zhouhan Lin, Minwei Feng, Cicero Nogueira dos Santos, Mo Yu, Bing Xiang, Bowen\nZhou, and Yoshua Bengio. A structured self-attentive sentence embedding. arXiv preprint\narXiv:1703.03130, 2017.\n[20] Samy Bengio Łukasz Kaiser. Can active memory replace attention? In Advances in Neural\nInformation Processing Systems, (NIPS), 2016.\n10',
  'metadata': {'publisher': 'Curran Associates, Inc.',
   'editors': 'I. Guyon and U.V. Luxburg and S. Bengio and H. Wallach and R. Fergus and S. Vishwanathan and R. Garnett',
   'page': 9,
   'creator': 'PyPDF',
   'date': '2017

### Integration Vectordb Context pipeline With LLM output


In [57]:
### Simple RAG pipeline with Groq LLM
from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv
load_dotenv()

### Initialize the Groq LLM (set your GROQ_API_KEY in environment)
groq_api_key = os.getenv("GROQ_API_KEY")

llm=ChatGroq(groq_api_key=groq_api_key,model_name="llama-3.1-8b-instant",temperature=0.1,max_tokens=1024)

## 2. Simple RAG function: retrieve context + generate response
def rag_simple(query,retriever,llm,top_k=3):
    ## retriever the context
    results=retriever.retrieve(query,top_k=top_k)
    context="\n\n".join([doc['content'] for doc in results]) if results else ""
    if not context:
        return "No relevant context found to answer the question."
    
    ## generate the answwer using GROQ LLM
    prompt=f"""Use the following context to answer the question concisely.
        Context:
        {context}

        Question: {query}

        Answer:"""
    
    response=llm.invoke([prompt.format(context=context,query=query)])
    return response.content

In [58]:
answer=rag_simple("will ml come in exam",rag_retriever,llm)
print(answer)

Retrieving documents for query: 'will ml come in exam'
Top K: 3, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 22.02it/s]

Generated embeddings with shape: (1, 384)
{'ids': [['doc_841e7e05_49', 'doc_16e092b8_49', 'doc_da97572a_8']], 'embeddings': None, 'documents': [['Exam Hack 🎯\nIf they ask:\n"What is Machine Learning?"\nAlways write:\nDefinition\nTypes of ML\nApplications\nAdvantages\nEven if not explicitly asked.\nProfs love it.\nUltra-Short Revision ⚡\nMachine Learning\n=\nLearning from Data\nTypes:\n1. Supervised\n2. Unsupervised\n3. Reinforcement\nNow only Perception and Action is left in Unit 4.\nGood news: it\'s one of the shortest topics in the entire syllabus and can be \nfinished in 10 minutes. 🚀\nlessgo', 'Exam Hack 🎯\nIf they ask:\n"What is Machine Learning?"\nAlways write:\nDefinition\nTypes of ML\nApplications\nAdvantages\nEven if not explicitly asked.\nProfs love it.\nUltra-Short Revision ⚡\nMachine Learning\n=\nLearning from Data\nTypes:\n1. Supervised\n2. Unsupervised\n3. Reinforcement\nNow only Perception and Action is left in Unit 4.\nGood news: it\'s one of the shortest topics in the 

Yes (1) or No (0). 

But in real life, things are not always black and white.


### Advance Rag Enhanced

In [65]:
# --- Enhanced RAG Pipeline Features ---
def rag_advanced(query, retriever, llm, top_k=5, min_score=0.2, return_context=False):
    """
    RAG pipeline with extra features:
    - Returns answer, sources, confidence score, and optionally full context.
    """
    results = retriever.retrieve(query, top_k=top_k, score_threshold=min_score)
    if not results:
        return {'answer': 'No relevant context found.', 'sources': [], 'confidence': 0.0, 'context': ''}
    
    # Prepare context and sources
    context = "\n\n".join([doc['content'] for doc in results])
    sources = [{
    'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
    'page': doc['metadata'].get('page', 'unknown'),
    'distance': doc['distance'],
    'preview': doc['content'][:300] + '...'
} for doc in results]

    # Smaller distance = better match
    confidence = 1 / (1 + min(doc['distance'] for doc in results))
    
    # Generate answer
    prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {query}\n\nAnswer:"""
    response = llm.invoke([prompt.format(context=context, query=query)])
    
    output = {
        'answer': response.content,
        'sources': sources,
        'confidence': confidence
    }
    if return_context:
        output['context'] = context
    return output

# Example usage:


In [67]:
result = rag_advanced("Hard Negetive Mining Techniques?", rag_retriever, llm, top_k=3, min_score=0.1, return_context=True)
print("Answer:", result['answer'])
print("Sources:", result['sources'])
print("Confidence:", result['confidence'])
print("Context Preview:", result['context'][:300])

Retrieving documents for query: 'Hard Negetive Mining Techniques?'
Top K: 3, Score threshold: 0.1
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00,  5.29it/s]

Generated embeddings with shape: (1, 384)
{'ids': [['doc_9a3e909b_38', 'doc_089ffa86_38', 'doc_688adfc4_44']], 'embeddings': None, 'documents': [['4. Easy Implementation\nWidely used.\nDisadvantages\n1. Need K in Advance\nMust know number of clusters.\n2. Sensitive to Initial Centroids\nDifferent starting points can give different answers.\n3. Poor with Outliers\nExtreme values affect results.\n4. Assumes Circular Clusters\nNot ideal for irregular shapes.\nDifference Between Classification and Clustering\nClassification Clustering\nSupervised Unsupervised\nLabels available No labels\nPredict class Discover groups\nCat/Dog Group similar objects\nVery common theory question.', '4. Easy Implementation\nWidely used.\nDisadvantages\n1. Need K in Advance\nMust know number of clusters.\n2. Sensitive to Initial Centroids\nDifferent starting points can give different answers.\n3. Poor with Outliers\nExtreme values affect results.\n4. Assumes Circular Clusters\nNot ideal for irregular shapes.\nD

Answer: The context provided does not mention "Hard Negative Mining Techniques." However, based on the information given, it seems like you are looking for a topic related to machine learning or data analysis. 

If you are looking for a technique that is the opposite of clustering, you might be thinking of "Negative Mining" or "Negative Sampling" which is a technique used in machine learning to reduce the impact of negative examples or outliers.
Sources: [{'source': 'U2 and U4.pdf', 'page': 37, 'distance': 1.351144790649414, 'preview': '4. Easy Implementation\nWidely used.\nDisadvantages\n1. Need K in Advance\nMust know number of clusters.\n2. Sensitive to Initial Centroids\nDifferent starting points can give different answers.\n3. Poor with Outliers\nExtreme values affect results.\n4. Assumes Circular Clusters\nNot ideal for irregular shape...'}, {'source': 'U2 and U4.pdf', 'page': 37, 'distance': 1.351144790649414, 'preview': '4. Easy Implementation\nWidely used.\nDisadvantages\n1. N